In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os

# point this at the directory *containing* tpch/
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))  
sys.path.insert(0, tpch_parent)

#from tpch import load_lineitem, load_orders, load_customer,  q03
import pandas as pd
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}  # or load from JSON

In [4]:

### cell 0 ###

# load just the datasets q01 needs:
def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


Index(['L_ORDERKEY', 'L_PARTKEY', 'L_SUPPKEY', 'L_LINENUMBER', 'L_QUANTITY',
       'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX', 'L_RETURNFLAG',
       'L_LINESTATUS', 'L_SHIPDATE', 'L_COMMITDATE', 'L_RECEIPTDATE',
       'L_SHIPINSTRUCT', 'L_SHIPMODE', 'L_COMMENT', 'L_DUMMY'],
      dtype='object')


In [5]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)


In [6]:

### cell 2 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [7]:
# %%time
# ### cell 3 ###

# # Define the cutoff date as a pandas Timestamp (will be handled on GPU)
# date = pd.Timestamp("1995-03-04")

# # 1) Filter rows and select only the needed columns, all on GPU
# fcustomer = customer.loc[
#     customer.C_MKTSEGMENT == "HOUSEHOLD",
#     ["C_CUSTKEY"]
# ]

# forders = orders.loc[
#     orders.O_ORDERDATE < date,
#     ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]
# ]

# flineitem = lineitem.loc[
#     lineitem.L_SHIPDATE > date,
#     ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
# ]

# # 2) Perform the two joins entirely on GPU
# jn2 = (
#     fcustomer
#       .merge(forders,    left_on="C_CUSTKEY", right_on="O_CUSTKEY")
#       .merge(flineitem,  left_on="O_ORDERKEY", right_on="L_ORDERKEY")
# )

# # 3) Compute the TMP column, group, sum and sort—all GPU operations
# jn2["TMP"] = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT)

# res = (
#     jn2
#       .groupby(
#           ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"],
#           as_index=False,
#           sort=False
#       )["TMP"]
#       .sum()
#       .sort_values("TMP", ascending=False)
# )

# # res now has the same schema [L_ORDERKEY, TMP, O_ORDERDATE, O_SHIPPRIORITY]


In [8]:
%%time 
#original
date = pd.Timestamp("1995-03-04")
lineitem_filtered = lineitem.loc[
    :, ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT", "L_SHIPDATE"]
]
orders_filtered = orders.loc[
    :, ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]
]
customer_filtered = customer.loc[:, ["C_MKTSEGMENT", "C_CUSTKEY"]]
lsel = lineitem_filtered.L_SHIPDATE > date
osel = orders_filtered.O_ORDERDATE < date
csel = customer_filtered.C_MKTSEGMENT == "HOUSEHOLD"
flineitem = lineitem_filtered[lsel]
forders = orders_filtered[osel]
fcustomer = customer_filtered[csel]
jn1 = fcustomer.merge(forders, left_on="C_CUSTKEY", right_on="O_CUSTKEY")
jn2 = jn1.merge(flineitem, left_on="O_ORDERKEY", right_on="L_ORDERKEY")
jn2["TMP"] = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT)
total = (
    jn2.groupby(
        ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"], as_index=False, sort=False
    )["TMP"]
    .sum()
    .sort_values(["TMP"], ascending=False)
)
res = total.loc[:, ["L_ORDERKEY", "TMP", "O_ORDERDATE", "O_SHIPPRIORITY"]]


CPU times: user 777 ms, sys: 499 ms, total: 1.28 s
Wall time: 1.22 s


In [11]:
res

,L_ORDERKEY,TMP,O_ORDERDATE,O_SHIPPRIORITY
106857,21679558,451192.3341,1995-01-17,0
56048,57978848,450250.4648,1995-02-19,0
86883,24295076,437186.7777,1995-01-05,0
9048,14767623,434238.9498,1995-02-17,0
18364,22488032,433122.3678,1995-02-18,0
...,...,...,...,...
13713,11388065,910.9464,1995-01-24,0
9656,17996129,904.8993,1994-11-24,0
102466,59991431,877.1945,1994-11-12,0
44064,37091424,863.0640,1995-01-10,0


In [9]:
res

,L_ORDERKEY,TMP,O_ORDERDATE,O_SHIPPRIORITY
106857,21679558,451192.3341,1995-01-17,0
56048,57978848,450250.4648,1995-02-19,0
86883,24295076,437186.7777,1995-01-05,0
9048,14767623,434238.9498,1995-02-17,0
18364,22488032,433122.3678,1995-02-18,0
...,...,...,...,...
13713,11388065,910.9464,1995-01-24,0
9656,17996129,904.8993,1994-11-24,0
102466,59991431,877.1945,1994-11-12,0
44064,37091424,863.0640,1995-01-10,0


In [10]:
### cell 4 ###

res.head()

,L_ORDERKEY,TMP,O_ORDERDATE,O_SHIPPRIORITY
106857,21679558,451192.3341,1995-01-17,0
56048,57978848,450250.4648,1995-02-19,0
86883,24295076,437186.7777,1995-01-05,0
9048,14767623,434238.9498,1995-02-17,0
18364,22488032,433122.3678,1995-02-18,0
